# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 11

Version 10 completed the first small decoder-only Transformer by stacking two reusable causal Transformer blocks.

Version 11 keeps the same decoder-only architecture and changes how text enters the model.

Instead of training on a tiny hardcoded character corpus, this version uses the public **WikiText-103 Raw v1** dataset and introduces **subword tokenization with Byte Pair Encoding (BPE)**.

The progression is:

text from WikiText-103  
→ BPE subword tokens  
→ integer token identifiers  
→ fixed-length token sequences  
→ token + positional embeddings  
→ Transformer block 1  
→ Transformer block 2  
→ vocabulary logits  
→ next-token prediction

The original training, validation and test split boundaries are preserved.

To keep the experiment small, computationally manageable and focused on tokenization, this version uses controlled subsets selected independently from each official dataset split while preserving the same inspectable decoder-only Transformer structure.

## 1. Imports and Configuration

Version 11 keeps TensorFlow for the decoder-only Transformer and NumPy for numerical inspection and validation.

Pandas is introduced to read the WikiText-103 Parquet files, while the `tokenizers` library is used to train and apply a Byte Pair Encoding tokenizer.

The configuration collects the values that control the experiment.

`TRAIN_ROWS`, `VALIDATION_ROWS` and `TEST_ROWS` define how many rows are used from the official WikiText-103 training, validation and test splits.

`VOCABULARY_SIZE` defines the maximum number of subword tokens learned by the BPE tokenizer.

`CONTEXT_LENGTH` defines the number of token positions processed together.

`MAX_TRAIN_SEQUENCES`, `MAX_VALIDATION_SEQUENCES` and `MAX_TEST_SEQUENCES` limit the number of fixed-length token sequences used by the model from each official dataset split, keeping the experiment computationally small and reproducible.

`EMBEDDING_DIM` defines the size of the trainable token and positional representations and therefore the main representation size of the model.

`ATTENTION_DIM` defines the size of the query, key and value vectors inside each Transformer block.

`FEED_FORWARD_DIM` defines the internal size of the position-wise feed-forward network.

`NUM_BLOCKS` defines how many Transformer blocks are stacked sequentially.

`LAYER_NORM_EPSILON` provides numerical stability during normalization.

`BATCH_SIZE`, `LEARNING_RATE` and `EPOCHS` control the optimization process.

An explicit seed controls weight initialization, dataset sampling and mini-batch shuffling, improving reproducibility across executions. Small numerical differences may still occur across software environments.

The main change in Version 11 is not the internal Transformer architecture, but the input representation: character identifiers are replaced by BPE subword token identifiers obtained from a documented public dataset.

In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import numpy as np
import pandas as pd
import tensorflow as tf

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from tokenizers.decoders import BPEDecoder

In [2]:
SEED = 42

TRAIN_ROWS = 10000
VALIDATION_ROWS = 1000
TEST_ROWS = 1000

VOCABULARY_SIZE = 2048
CONTEXT_LENGTH = 16

MAX_TRAIN_SEQUENCES = 12000
MAX_VALIDATION_SEQUENCES = 2000
MAX_TEST_SEQUENCES = 2000

BATCH_SIZE = 32
LEARNING_RATE = 1.0
EPOCHS = 20

EMBEDDING_DIM = 8
ATTENTION_DIM = 16
FEED_FORWARD_DIM = 16
NUM_BLOCKS = 2
LAYER_NORM_EPSILON = 1e-5

tf.keras.utils.set_random_seed(SEED)

## 2. Public Dataset

Version 11 replaces the hardcoded 440-character corpus with **WikiText-103 Raw v1**, a public language-modeling dataset based on verified Good and Featured Wikipedia articles.

The dataset is attached to the Kaggle notebook as an input and contains the original training, validation and test splits in Parquet format.

The files are:

- two Parquet shards for the training split;
- one Parquet file for the validation split;
- one Parquet file for the test split.

Each file contains a single `text` column.

The original split boundaries are preserved. Version 11 does not create a new random train/validation split.

To keep the experiment small and inspectable, a reproducible subset is selected independently from each official split using the row limits defined in the configuration.

Empty rows are removed before sampling because WikiText contains blank entries that represent document and section boundaries rather than useful standalone training text.

The dataset language is English. Source, provenance and licensing information are documented in the attached Kaggle dataset metadata for the `wikitext-103-raw-v1` configuration.

In [3]:
DATASET_DIRECTORY = "/kaggle/input/datasets/lucalullo/wiki-text-103-raw-v1"

train_files = [
    os.path.join(DATASET_DIRECTORY, "train-00000-of-00002.parquet"),
    os.path.join(DATASET_DIRECTORY, "train-00001-of-00002.parquet"),
]

validation_file = os.path.join(DATASET_DIRECTORY, "validation-00000-of-00001.parquet")
test_file = os.path.join(DATASET_DIRECTORY, "test-00000-of-00001.parquet")

for path in train_files + [validation_file, test_file]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

print("Dataset files found:")
for path in train_files + [validation_file, test_file]:
    print(os.path.basename(path))

Dataset files found:
train-00000-of-00002.parquet
train-00001-of-00002.parquet
validation-00000-of-00001.parquet
test-00000-of-00001.parquet


In [4]:
train_data = pd.concat(
    [pd.read_parquet(path, columns=["text"]) for path in train_files],
    ignore_index=True
)

validation_data = pd.read_parquet(validation_file, columns=["text"])
test_data = pd.read_parquet(test_file, columns=["text"])

print("Original rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Original rows:
Train: 1801350
Validation: 3760
Test: 4358


In [5]:
def prepare_text_subset(data, number_of_rows, seed):
    cleaned = data.dropna(subset=["text"]).copy()
    cleaned = cleaned[cleaned["text"].str.strip().str.len() > 0]
    return cleaned.sample(n=min(number_of_rows, len(cleaned)), random_state=seed).reset_index(drop=True)

train_data = prepare_text_subset(train_data, TRAIN_ROWS, SEED)
validation_data = prepare_text_subset(validation_data, VALIDATION_ROWS, SEED)
test_data = prepare_text_subset(test_data, TEST_ROWS, SEED)

print("Experiment rows:")
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("\nExample training text:")
print(train_data.loc[0, "text"])

Experiment rows:
Train: 10000
Validation: 1000
Test: 1000

Example training text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 



## 3. Subword Tokenizer

Version 11 replaces character-level tokenization with **Byte Pair Encoding (BPE)**.

BPE starts from small text units and repeatedly merges frequent neighboring units to build a vocabulary of reusable subword tokens.

The tokenizer is trained only on the selected training subset. Validation and test text are never used to construct the vocabulary, preserving the separation between the official dataset splits.

The `</w>` suffix marks the end of a pre-tokenized word, allowing the decoder to reconstruct word boundaries after BPE tokenization.

`VOCABULARY_SIZE` defines the maximum number of tokens that the tokenizer can learn.

A special `[UNK]` token represents text units that cannot be mapped to the learned vocabulary.

After training, the tokenizer converts text into integer token identifiers that can be processed by the decoder-only Transformer.

In [6]:
tokenizer = Tokenizer(BPE(
    unk_token="[UNK]",
    end_of_word_suffix="</w>"
))

tokenizer.pre_tokenizer = Whitespace()
tokenizer.decoder = BPEDecoder(suffix="</w>")

trainer = BpeTrainer(
    vocab_size=VOCABULARY_SIZE,
    min_frequency=2,
    special_tokens=["[UNK]"],
    end_of_word_suffix="</w>"
)

tokenizer.train_from_iterator(train_data["text"], trainer=trainer)

vocabulary_size = tokenizer.get_vocab_size()

print("Vocabulary size:", vocabulary_size)




Vocabulary size: 2048


### Inspecting the Learned Tokens

The tokenizer can now represent text using variable-length subword units instead of individual characters.

Frequent words or word fragments may become single tokens, while less frequent words can be decomposed into multiple smaller tokens.

In [7]:
example_text = train_data.loc[0, "text"]
encoded_example = tokenizer.encode(example_text)

print("Text:")
print(example_text)

print("\nTokens:")
print(encoded_example.tokens[:30])

print("\nToken IDs:")
print(encoded_example.ids[:30])

print("\nNumber of characters:", len(example_text))
print("Number of tokens:", len(encoded_example.ids))

Text:
 In cooperation with various nonprofit organizations , since 2008 the company has organized the THSRC Smile Program , which offers minority groups and families the chance to experience the comfort and convenience of high @-@ speed rail rides . THSRC Smile Program gave its first ride on 23 January 2008 to children , social workers and parents from the Fushan branch of the Taiwan Fund for Children and Families . As of the end of 2011 , 12 @,@ 231 people in total , comprising 229 underprivileged groups and families , have participated in the THSRC Smile Program . 


Tokens:
['In</w>', 'co', 'oper', 'ation</w>', 'with</w>', 'vari', 'ous</w>', 'n', 'on', 'pro', 'f', 'it</w>', 'org', 'ani', 'z', 'ations</w>', ',</w>', 'since</w>', '200', '8</w>', 'the</w>', 'comp', 'any</w>', 'has</w>', 'org', 'ani', 'zed</w>', 'the</w>', 'T', 'H']

Token IDs:
[1146, 1228, 1730, 1103, 1066, 1919, 1272, 78, 999, 1082, 70, 1102, 1875, 1735, 90, 1434, 898, 1927, 1196, 694, 991, 1452, 1389, 1260, 1875, 173

## 4. Token Sequences

The decoder-only Transformer still learns next-token prediction at every context position.

Each WikiText row is encoded independently with the trained BPE tokenizer.

Token sequences are divided into fixed-length windows of `CONTEXT_LENGTH + 1` tokens. The first `CONTEXT_LENGTH` tokens form the input and the same sequence shifted by one position forms the target.

For example:

input tokens  → `[t0, t1, t2, ..., t15]`  
target tokens → `[t1, t2, t3, ..., t16]`

Unlike the character-level examples used in earlier versions, the windows now contain subword token identifiers.

Windows are created independently inside each text row so examples never cross row boundaries. A stride equal to `CONTEXT_LENGTH` avoids excessive overlap and keeps the experiment compact.

In [8]:
def create_token_sequences(texts, tokenizer, context_length):
    inputs, targets = [], []

    for text in texts:
        token_ids = tokenizer.encode(text).ids

        for start in range(0, len(token_ids) - context_length, context_length):
            sequence = token_ids[start:start + context_length + 1]

            if len(sequence) == context_length + 1:
                inputs.append(sequence[:-1])
                targets.append(sequence[1:])

    return tf.constant(inputs, dtype=tf.int32), tf.constant(targets, dtype=tf.int32)

In [9]:
train_inputs, train_targets = create_token_sequences(train_data["text"], tokenizer, CONTEXT_LENGTH)
validation_inputs, validation_targets = create_token_sequences(validation_data["text"], tokenizer, CONTEXT_LENGTH)
test_inputs, test_targets = create_token_sequences(test_data["text"], tokenizer, CONTEXT_LENGTH)

print("Train:", train_inputs.shape, train_targets.shape)
print("Validation:", validation_inputs.shape, validation_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

print("\nFirst input IDs:")
print(train_inputs[0].numpy())

print("\nFirst target IDs:")
print(train_targets[0].numpy())

print("\nDecoded input:")
print(tokenizer.decode(train_inputs[0].numpy()))

Train: (96893, 16) (96893, 16)
Validation: (9764, 16) (9764, 16)
Test: (9572, 16) (9572, 16)

First input IDs:
[1146 1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735
   90 1434]

First target IDs:
[1228 1730 1103 1066 1919 1272   78  999 1082   70 1102 1875 1735   90
 1434  898]

Decoded input:
In cooperation with various nonprofit organizations


In [10]:
train_inputs = train_inputs[:MAX_TRAIN_SEQUENCES]
train_targets = train_targets[:MAX_TRAIN_SEQUENCES]

validation_inputs = validation_inputs[:MAX_VALIDATION_SEQUENCES]
validation_targets = validation_targets[:MAX_VALIDATION_SEQUENCES]

test_inputs = test_inputs[:MAX_TEST_SEQUENCES]
test_targets = test_targets[:MAX_TEST_SEQUENCES]

## 5. Neural Model

The decoder-only Transformer architecture remains the same as in Version 10.

The important change is at the input and output boundaries.

Character embeddings are replaced by **token embeddings**, and the embedding matrix now contains one trainable vector for every token in the BPE vocabulary.

The positional embedding matrix also grows from four positions to `CONTEXT_LENGTH` token positions.

Each Transformer block still contains:

- query, key and value projections;
- causal scaled dot-product self-attention;
- an attention output projection;
- a residual connection followed by LayerNorm;
- a position-wise feed-forward network;
- a second residual connection followed by LayerNorm.

Two independent Transformer blocks are stacked sequentially.

The final projection maps the model representation at every token position to logits over the complete BPE vocabulary.

In [11]:
def layer_normalize(values, scale, shift):
    mean = tf.reduce_mean(values, axis=-1, keepdims=True)
    variance = tf.reduce_mean(tf.square(values - mean), axis=-1, keepdims=True)
    normalized = (values - mean) / tf.sqrt(variance + LAYER_NORM_EPSILON)
    return normalized * scale + shift


class TransformerBlock:
    def __init__(self, random_generator):
        self.query_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02))
        self.key_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02))
        self.value_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, ATTENTION_DIM), stddev=0.02))
        self.attention_output_weights = tf.Variable(random_generator.normal((ATTENTION_DIM, EMBEDDING_DIM), stddev=0.02))

        self.layer_norm_1_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)))
        self.layer_norm_1_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)))

        self.feed_forward_input_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, FEED_FORWARD_DIM), stddev=0.02))
        self.feed_forward_output_weights = tf.Variable(random_generator.normal((FEED_FORWARD_DIM, EMBEDDING_DIM), stddev=0.02))

        self.layer_norm_2_scale = tf.Variable(tf.ones((EMBEDDING_DIM,)))
        self.layer_norm_2_shift = tf.Variable(tf.zeros((EMBEDDING_DIM,)))

    @property
    def trainable_variables(self):
        return [
            self.query_weights, self.key_weights, self.value_weights,
            self.attention_output_weights,
            self.layer_norm_1_scale, self.layer_norm_1_shift,
            self.feed_forward_input_weights, self.feed_forward_output_weights,
            self.layer_norm_2_scale, self.layer_norm_2_shift
        ]

    def __call__(self, values):
        queries = tf.matmul(values, self.query_weights)
        keys = tf.matmul(values, self.key_weights)
        values_projected = tf.matmul(values, self.value_weights)

        scores = tf.matmul(queries, keys, transpose_b=True)
        scores = scores / tf.sqrt(tf.cast(ATTENTION_DIM, tf.float32))

        sequence_length = tf.shape(scores)[-1]
        causal_mask = tf.linalg.band_part(tf.ones((sequence_length, sequence_length)), -1, 0)
        masked_scores = tf.where(causal_mask == 1, scores, tf.constant(-1e9, dtype=tf.float32))

        attention_weights = tf.nn.softmax(masked_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, values_projected)
        attention_output = tf.matmul(attention_output, self.attention_output_weights)

        values = layer_normalize(
            values + attention_output,
            self.layer_norm_1_scale,
            self.layer_norm_1_shift
        )

        feed_forward = tf.nn.relu(tf.matmul(values, self.feed_forward_input_weights))
        feed_forward = tf.matmul(feed_forward, self.feed_forward_output_weights)

        output = layer_normalize(
            values + feed_forward,
            self.layer_norm_2_scale,
            self.layer_norm_2_shift
        )

        return output, attention_weights


class DecoderOnlyLanguageModel:
    def __init__(self, vocabulary_size, seed=42):
        random_generator = tf.random.Generator.from_seed(seed)

        self.token_embeddings = tf.Variable(
            random_generator.normal((vocabulary_size, EMBEDDING_DIM), stddev=0.02)
        )
        self.positional_embeddings = tf.Variable(
            random_generator.normal((CONTEXT_LENGTH, EMBEDDING_DIM), stddev=0.02)
        )

        self.blocks = [TransformerBlock(random_generator) for _ in range(NUM_BLOCKS)]

        self.output_weights = tf.Variable(random_generator.normal((EMBEDDING_DIM, vocabulary_size), stddev=0.02))

    @property
    def trainable_variables(self):
        variables = [self.token_embeddings, self.positional_embeddings]

        for block in self.blocks:
            variables.extend(block.trainable_variables)

        variables.append(self.output_weights)
        return variables

    def __call__(self, input_ids):
        token_values = tf.gather(self.token_embeddings, input_ids)
        values = token_values + self.positional_embeddings
        attention_weights = []

        for block in self.blocks:
            values, block_attention = block(values)
            attention_weights.append(block_attention)

        logits = tf.matmul(values, self.output_weights)
        return values, attention_weights, logits

In [12]:
vocabulary_size = tokenizer.get_vocab_size()

model = DecoderOnlyLanguageModel(vocabulary_size=vocabulary_size, seed=SEED)

print("Tokenizer vocabulary size:", vocabulary_size)
print("Token embedding shape:", model.token_embeddings.shape)

Tokenizer vocabulary size: 2048
Token embedding shape: (2048, 8)


### Model Structure and Trainable Parameters

The decoder-only Transformer now operates on BPE token sequences instead of character identifiers.

The tokenizer uses a vocabulary of 2048 tokens and every input sequence contains 16 token positions.

The Transformer architecture itself remains unchanged: two independent causal Transformer blocks process the token and positional embeddings before the final projection produces logits over the complete BPE vocabulary.

Inspecting the tensor shapes and parameter count confirms that the new token representation is connected correctly to the model.

In [13]:
example_input = train_inputs[:1]

model_output, attention_weights, logits = model(example_input)

total_parameters = sum(int(np.prod(variable.shape)) for variable in model.trainable_variables)

print("Vocabulary size:", vocabulary_size)
print("Number of Transformer blocks:", len(model.blocks))
print("Number of trainable variables:", len(model.trainable_variables))

print("\nInput shape:", example_input.shape)
print("Model output shape:", model_output.shape)
print("Logits shape:", logits.shape)

for index, block_attention in enumerate(attention_weights, start=1):
    print(f"Block {index} attention shape:", block_attention.shape)

print("\nTotal trainable parameters:", total_parameters)

Vocabulary size: 2048
Number of Transformer blocks: 2
Number of trainable variables: 23

Input shape: (1, 16)
Model output shape: (1, 16, 8)
Logits shape: (1, 16, 2048)
Block 1 attention shape: (1, 16, 16)
Block 2 attention shape: (1, 16, 16)

Total trainable parameters: 34496


## 6. Training

Training still uses next-token prediction with sparse categorical cross-entropy.

For every input sequence of 16 BPE tokens, the model predicts the following token at each position.

Because Version 11 contains substantially more training examples and a much larger vocabulary than the previous character-level model, loss computation is performed in mini-batches rather than by passing the complete dataset through the model at once.

The training examples are shuffled with a fixed seed at every epoch, while validation examples remain unchanged.

Gradient descent updates all trainable variables of the token embeddings, positional embeddings, Transformer blocks and final vocabulary projection.

In [14]:
def compute_batch_loss(model, inputs, targets):
    _, _, logits = model(inputs)

    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=targets, logits=logits)

    return tf.reduce_mean(losses)

In [15]:
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((train_inputs, train_targets))
    .shuffle(
        buffer_size=len(train_inputs),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
)

validation_dataset = (tf.data.Dataset.from_tensor_slices((validation_inputs, validation_targets)).batch(BATCH_SIZE))

test_dataset = (tf.data.Dataset.from_tensor_slices((test_inputs, test_targets)).batch(BATCH_SIZE))

print("Training sequences:", len(train_inputs))
print("Validation sequences:", len(validation_inputs))
print("Test sequences:", len(test_inputs))

print("Training batches:", len(train_dataset))
print("Validation batches:", len(validation_dataset))
print("Test batches:", len(test_dataset))

Training sequences: 12000
Validation sequences: 2000
Test sequences: 2000
Training batches: 375
Validation batches: 63
Test batches: 63


In [16]:
def compute_dataset_loss(model, dataset):
    total_loss = 0.0
    total_tokens = 0

    for batch_inputs, batch_targets in dataset:
        batch_loss = compute_batch_loss(
            model,
            batch_inputs,
            batch_targets
        )

        batch_tokens = int(tf.size(batch_targets))

        total_loss += float(batch_loss) * batch_tokens
        total_tokens += batch_tokens

    return total_loss / total_tokens

In [17]:
initial_train_loss = compute_dataset_loss(model, train_dataset)
initial_validation_loss = compute_dataset_loss(model, validation_dataset)

random_baseline = np.log(vocabulary_size)

print(f"Initial training loss: {initial_train_loss:.6f}")
print(f"Initial validation loss: {initial_validation_loss:.6f}")
print(f"Random baseline: {random_baseline:.6f}")

Initial training loss: 7.626767
Initial validation loss: 7.626187
Random baseline: 7.624619


### Gradient Descent

Each mini-batch performs one gradient-descent update.

TensorFlow's `GradientTape` records the operations used to compute the batch loss and obtains the gradient of that loss with respect to every trainable variable.

The parameters are then updated explicitly using the configured learning rate.

The mean training loss is accumulated from the mini-batches processed during each epoch, while validation loss is computed separately without modifying the model parameters.

Token embedding lookups can produce sparse `IndexedSlices` gradients. These are updated with `scatter_sub`, while dense gradients use the standard `assign_sub` update.

In [18]:
def apply_gradient(variable, gradient, learning_rate):
    if gradient is None:
        return

    if isinstance(gradient, tf.IndexedSlices):
        update = tf.IndexedSlices(
            values=learning_rate * gradient.values,
            indices=gradient.indices,
            dense_shape=gradient.dense_shape
        )
        variable.scatter_sub(update)

    else:
        variable.assign_sub(learning_rate * gradient)

In [19]:
def train_model(model, train_dataset, validation_dataset, epochs, learning_rate):
    training_losses = []
    validation_losses = []

    for epoch in range(1, epochs + 1):
        total_training_loss = 0.0
        total_training_tokens = 0

        for batch_inputs, batch_targets in train_dataset:
            with tf.GradientTape() as tape:
                batch_loss = compute_batch_loss(
                    model,
                    batch_inputs,
                    batch_targets
                )

            gradients = tape.gradient(
                batch_loss,
                model.trainable_variables
            )

            for variable, gradient in zip(
                model.trainable_variables,
                gradients
            ):
                apply_gradient(
                    variable,
                    gradient,
                    learning_rate
                )

            batch_tokens = int(tf.size(batch_targets))

            total_training_loss += float(batch_loss) * batch_tokens
            total_training_tokens += batch_tokens

        training_loss = total_training_loss / total_training_tokens
        validation_loss = compute_dataset_loss(
            model,
            validation_dataset
        )

        training_losses.append(training_loss)
        validation_losses.append(validation_loss)

        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:3d} | "
                f"Train loss: {training_loss:.6f} | "
                f"Validation loss: {validation_loss:.6f}"
            )

    return training_losses, validation_losses

In [20]:
training_losses, validation_losses = train_model(
    model,
    train_dataset,
    validation_dataset,
    EPOCHS,
    LEARNING_RATE
)

Epoch   1 | Train loss: 5.999189 | Validation loss: 5.515175
Epoch  10 | Train loss: 5.089073 | Validation loss: 5.142680
Epoch  20 | Train loss: 5.021189 | Validation loss: 5.085637


### Final Losses

After training, the model is evaluated on the selected subsets of the three official dataset splits.

Training loss measures how well the final model predicts examples used during optimization.

Validation loss measures performance on unseen examples used only to inspect the learning process.

Test loss is computed only after training and provides a final measurement on the selected subset of the independent WikiText test split.

All three losses are calculated with the same mini-batch procedure.

In [21]:
final_train_loss = compute_dataset_loss(model, train_dataset)
final_validation_loss = compute_dataset_loss(model, validation_dataset)
test_loss = compute_dataset_loss(model, test_dataset)

print(f"Initial training loss:   {initial_train_loss:.6f}")
print(f"Initial validation loss: {initial_validation_loss:.6f}")

print(f"\nFinal training loss:     {final_train_loss:.6f}")
print(f"Final validation loss:   {final_validation_loss:.6f}")
print(f"Test loss:               {test_loss:.6f}")

print(f"\nRandom baseline:         {random_baseline:.6f}")

Initial training loss:   7.626767
Initial validation loss: 7.626187

Final training loss:     4.987922
Final validation loss:   5.085637
Test loss:               5.117153

Random baseline:         7.624619


## 7. Next-Token Prediction

The trained decoder-only Transformer produces one probability distribution over the BPE vocabulary at every context position.

For next-token prediction, the model receives a sequence of 16 tokens and the logits produced at the final position represent the model's prediction for the token that should follow the complete context.

Applying softmax converts these logits into probabilities.

Because the vocabulary now contains BPE subword units, the predictions may represent complete words, word fragments or punctuation rather than individual characters.

In [22]:
def predict_next_token_probabilities(model, input_ids):
    input_tensor = tf.constant([input_ids], dtype=tf.int32)

    _, _, logits = model(input_tensor)

    next_token_logits = logits[0, -1]
    probabilities = tf.nn.softmax(next_token_logits)

    return probabilities.numpy()

In [23]:
prediction_context = test_inputs[0].numpy()

print("Context token IDs:")
print(prediction_context)

print("\nContext tokens:")
print([tokenizer.id_to_token(int(token_id)) for token_id in prediction_context])

print("\nDecoded context:")
print(tokenizer.decode(prediction_context.tolist()))

Context token IDs:
[  44 1012 1708   39 1266 1124 1047  991   70 1217 1080 1003 1968 1071
   69  684]

Context tokens:
['L', 'it', 'tle</w>', 'G', 'id', 'ding</w>', 'is</w>', 'the</w>', 'f', 'our', 'th</w>', 'and</w>', 'final</w>', 'po', 'e', 'm</w>']

Decoded context:
Little Gidding is the fourth and final poem


In [24]:
probabilities = predict_next_token_probabilities(model, prediction_context)

top_indices = np.argsort(probabilities)[-10:][::-1]

print("Top next-token predictions:\n")

for token_id in top_indices:
    token = tokenizer.id_to_token(int(token_id))
    probability = probabilities[token_id]

    print(
        f"{token!r:20s} "
        f"id={token_id:4d} "
        f"probability={probability:.4f}"
    )

Top next-token predictions:

',</w>'              id= 898 probability=0.1392
'.</w>'              id= 772 probability=0.0825
'and</w>'            id=1003 probability=0.0471
'of</w>'             id=1004 probability=0.0453
"'</w>"              id= 725 probability=0.0336
'(</w>'              id= 974 probability=0.0253
'was</w>'            id=1029 probability=0.0252
')</w>'              id= 937 probability=0.0226
'"</w>'              id= 790 probability=0.0214
'in</w>'             id=1007 probability=0.0199


In [25]:
true_next_token_id = int(test_targets[0, -1])
true_next_token = tokenizer.id_to_token(true_next_token_id)

print("True next token:", repr(true_next_token))
print("True token ID:", true_next_token_id)
print("Model probability:", f"{probabilities[true_next_token_id]:.4f}")

True next token: 'of</w>'
True token ID: 1004
Model probability: 0.0453


## 8. Autoregressive Generation

The trained model can now generate text one BPE token at a time.

Generation begins from an initial text prompt. The prompt is encoded with the same BPE tokenizer used during training.

At every step:

1. the most recent `CONTEXT_LENGTH` token identifiers are passed to the Transformer;
2. the logits at the final position are converted into probabilities;
3. the token with the highest probability is selected;
4. the selected token is appended to the sequence;
5. the process repeats using the updated context.

This version uses greedy decoding, selecting the most probable token at every step.

Because the model has a context length of 16 tokens, the prompt must contain at least 16 BPE tokens. Once the sequence becomes longer, only the most recent 16 tokens are used as context.

Finally, the complete sequence of BPE token identifiers is decoded back into readable text.

In [26]:
def generate_text(model, tokenizer, prompt, number_of_tokens):
    generated_ids = tokenizer.encode(prompt).ids

    if len(generated_ids) < CONTEXT_LENGTH:
        raise ValueError(f"The prompt must contain at least {CONTEXT_LENGTH} tokens.")

    for _ in range(number_of_tokens):
        context_ids = generated_ids[-CONTEXT_LENGTH:]

        probabilities = predict_next_token_probabilities(model, context_ids)

        next_token_id = int(np.argmax(probabilities))
        generated_ids.append(next_token_id)

    return generated_ids, tokenizer.decode(generated_ids)

In [27]:
generation_prompt = tokenizer.decode(prediction_context.tolist())

generated_ids, generated_text = generate_text(
    model,
    tokenizer,
    generation_prompt,
    number_of_tokens=40
)

print("Prompt:")
print(generation_prompt)

print("\nGenerated text:")
print(generated_text)

Prompt:
Little Gidding is the fourth and final poem

Generated text:
Little Gidding is the fourth and final poem , and the Sea , and the Sea , and the Sea , and the Sea , and the Sea , and the Sea , and the S


In [28]:
prompt_length = len(tokenizer.encode(generation_prompt).ids)

new_token_ids = generated_ids[prompt_length:]

print("Generated BPE tokens:")
print([tokenizer.id_to_token(token_id) for token_id in new_token_ids])

Generated BPE tokens:
[',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S', 'e', 'a</w>', ',</w>', 'and</w>', 'the</w>', 'S']


## 9. Inspecting Attention

The Transformer still uses causal self-attention after the transition from characters to BPE tokens.

With a context length of 16, each Transformer block produces a `16 × 16` attention matrix for every example.

Rather than printing the complete matrices, the following inspection focuses on the final context position used for next-token prediction.

The attention weights show how strongly the final token attends to each earlier BPE token in the context.

Because the attention mechanism is causal, the final position may attend to every token in the current context, while earlier positions cannot attend to future tokens.

In [29]:
attention_context = test_inputs[0:1]

_, attention_weights, _ = model(attention_context)

attention_tokens = [
    tokenizer.id_to_token(int(token_id))
    for token_id in attention_context[0].numpy()
]

print("Context:")
print(tokenizer.decode(attention_context[0].numpy().tolist()))

print("\nTokens:")
print(attention_tokens)

Context:
Little Gidding is the fourth and final poem

Tokens:
['L', 'it', 'tle</w>', 'G', 'id', 'ding</w>', 'is</w>', 'the</w>', 'f', 'our', 'th</w>', 'and</w>', 'final</w>', 'po', 'e', 'm</w>']


In [30]:
for block_index, block_attention in enumerate(attention_weights, start=1):
    final_position_attention = block_attention[0, -1].numpy()

    print(f"\nTransformer block {block_index}")
    print("-" * 45)

    for token, weight in zip(attention_tokens, final_position_attention):
        print(f"{token!r:18s} {weight:.4f}")


Transformer block 1
---------------------------------------------
'L'                0.0625
'it'               0.0625
'tle</w>'          0.0625
'G'                0.0625
'id'               0.0625
'ding</w>'         0.0625
'is</w>'           0.0625
'the</w>'          0.0625
'f'                0.0625
'our'              0.0625
'th</w>'           0.0625
'and</w>'          0.0625
'final</w>'        0.0625
'po'               0.0625
'e'                0.0625
'm</w>'            0.0625

Transformer block 2
---------------------------------------------
'L'                0.1621
'it'               0.0531
'tle</w>'          0.0290
'G'                0.2545
'id'               0.0350
'ding</w>'         0.0299
'is</w>'           0.0302
'the</w>'          0.0600
'f'                0.0861
'our'              0.0437
'th</w>'           0.0335
'and</w>'          0.0430
'final</w>'        0.0307
'po'               0.0400
'e'                0.0297
'm</w>'            0.0393


In [31]:
for block_index, block_attention in enumerate(attention_weights, start=1):
    final_position_attention = block_attention[0, -1].numpy()

    print(f"Block {block_index} attention sum:", f"{final_position_attention.sum():.6f}")

Block 1 attention sum: 1.000000
Block 2 attention sum: 1.000000


## 10. Results

Version 11 successfully replaces character-level input with BPE subword tokenization trained on a public language-modeling dataset.

The tokenizer learns a vocabulary of 2048 tokens from the selected WikiText-103 training data.

The decoder-only Transformer processes sequences of 16 BPE tokens using two causal Transformer blocks and contains 34,496 trainable parameters.

The saved execution above reports the initial and final training and validation losses, together with the final test loss. The reported losses allow training, validation and test performance to be compared with the uniform-vocabulary baseline while keeping the three dataset subsets separate.

Next-token inspection demonstrates how the trained model converts final-position logits into a probability distribution over BPE tokens on unseen test data.

Autoregressive generation demonstrates the complete token pipeline:

text  
→ BPE encoding  
→ token context  
→ Transformer prediction  
→ generated BPE tokens  
→ decoded text

Greedy decoding selects the most probable token at every step and is deterministic for a given trained model. In a deliberately small model with a limited context, this strategy can produce repetitive generation patterns.

Attention inspection confirms that the two Transformer blocks can develop different attention patterns while preserving the causal constraint.

## 11. Tests

The following checks verify the main structural and numerical properties of Version 11.

They confirm that token identifiers fit inside the BPE vocabulary, sequence shapes are correct, targets are shifted by one token, model output dimensions match the vocabulary, causal attention remains normalized and masked correctly, training improves the loss, and generation produces additional tokens.

In [32]:
assert vocabulary_size == 2048

assert train_inputs.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)
assert train_targets.shape == (MAX_TRAIN_SEQUENCES, CONTEXT_LENGTH)

assert validation_inputs.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)

assert test_inputs.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)

assert validation_targets.shape == (MAX_VALIDATION_SEQUENCES, CONTEXT_LENGTH)
assert test_targets.shape == (MAX_TEST_SEQUENCES, CONTEXT_LENGTH)

assert int(tf.reduce_max(train_inputs)) < vocabulary_size
assert int(tf.reduce_max(train_targets)) < vocabulary_size

assert np.array_equal(train_inputs[0, 1:].numpy(), train_targets[0, :-1].numpy())

assert model_output.shape == (1, CONTEXT_LENGTH, EMBEDDING_DIM)

assert logits.shape == (1, CONTEXT_LENGTH, vocabulary_size)

assert len(model.blocks) == NUM_BLOCKS
assert len(model.trainable_variables) == 23
assert total_parameters == 34496

for block_attention in attention_weights:
    matrix = block_attention[0].numpy()

    assert matrix.shape == ( CONTEXT_LENGTH, CONTEXT_LENGTH)

    assert np.allclose( matrix.sum(axis=-1), 1.0, atol=1e-6)

    assert np.allclose(np.triu(matrix, k=1), 0.0, atol=1e-6)

assert final_train_loss < initial_train_loss
assert final_validation_loss < initial_validation_loss
assert test_loss < random_baseline

assert len(generated_ids) > len(tokenizer.encode(generation_prompt).ids)

assert model.blocks[0] is not model.blocks[1]

assert len(training_losses) == EPOCHS
assert len(validation_losses) == EPOCHS

assert np.all(np.isfinite(training_losses))
assert np.all(np.isfinite(validation_losses))

assert np.isfinite(final_train_loss)
assert np.isfinite(final_validation_loss)
assert np.isfinite(test_loss)

assert np.all(np.isfinite(probabilities))
assert np.isclose(probabilities.sum(), 1.0, atol=1e-6)

assert final_train_loss < random_baseline
assert final_validation_loss < random_baseline

assert len(new_token_ids) == 40

print("All checks passed.")

All checks passed.


## Notes

- Version 11 changes the representation of language entering the decoder-only Transformer.

- The earlier character vocabulary has been replaced by a BPE tokenizer trained on a documented public dataset. Text is now represented using reusable subword units, allowing frequent words and fragments to occupy individual vocabulary entries while less frequent words can still be decomposed into smaller pieces.

- WikiText-103 Raw v1 provides separate training, validation and test data, replacing the tiny hardcoded corpus and the randomly created validation split.

- The Transformer architecture itself remains deliberately small and inspectable: two causal Transformer blocks operate on token and positional embeddings before projecting each position to logits over a 2048-token vocabulary.

- Training, validation and test losses are reported separately and can be compared with the uniform-vocabulary baseline while preserving the separation between the dataset subsets.

- The model can also perform next-token prediction and autoregressive generation directly in BPE token space. Generated subword tokens are decoded back into readable text, completing the transition from a character-level educational model to a small language model operating on public text and learned subword representations.

- The generated text is still limited by the small representation size, short context window, restricted training subset and greedy decoding strategy. These constraints are intentional: the purpose of this version is to isolate and make visible the transition from characters to public-data subword language modeling.

- Because whitespace pre-tokenization treats punctuation as separate units, decoded text may contain spaces before punctuation. This is a tokenizer-formatting limitation rather than a Transformer error.

Future versions will introduce new components and gradually evolve the architecture.